# Section-Aware RAG System with Romanian City Text + Qwen
## LangChain-based Retrieval-Augmented Generation

This notebook implements a metadata-aware RAG system:
- Loads Romanian Wikipedia-derived city text through `city_preprocessing`
- Normalizes and sections each city before chunking
- Builds one profile document plus one document per useful section
- Chunks with `RecursiveCharacterTextSplitter` after sectioning
- Stores section/city metadata in Chroma for filtered retrieval
- Uses Qwen for grounded Romanian answers

**Prerequisites:**
- Local server running on http://localhost:1234/v1 (for both embeddings and LLM)
- City text files in `informations/cities_text/` or `cities_text/`
- Installed: langchain, chroma, openai, langchain-text-splitters


In [1]:
## Pasul 1: Instaleaza bibliotecile necesare

import subprocess
import sys

packages = ['langchain', 'langchain-openai', 'chromadb', 'langchain-text-splitters']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} deja instalat")
    except ImportError:
        print(f"⚠ Se instaleaza {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f"✓ {package} instalat")

print("\n✓ Toate dependintele sunt gata!")

✓ langchain deja instalat
⚠ Se instaleaza langchain-openai...
✓ langchain-openai instalat
⚠ Se instaleaza chromadb...
✓ chromadb instalat


c:\Users\Alex\Documents\llms\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ langchain-text-splitters deja instalat

✓ Toate dependintele sunt gata!


In [4]:
## Pasul 2: Incarca documentele curate si section-aware ale oraselor

from pathlib import Path
from city_preprocessing.pipeline import load_city_documents

print("=" * 100)
print("INCARCARE DOCUMENTE SECTION-AWARE ALE ORASELOR")
print("=" * 100)

text_dir = Path('informations/cities_text')
if not text_dir.exists():
    text_dir = Path('cities_text')

print(f"\n📁 Se incarca din: {text_dir}")
docs = load_city_documents(text_dir)

print(f"\n✓ Incarcat {len(docs)} documente RAG curate")
print("   (un Profil general per oras + cate un document per sectiune utila)")

cities = sorted({doc.metadata.get('city_key') for doc in docs})
sections = sorted({doc.metadata.get('section') for doc in docs})
total_chars = sum(len(doc.page_content) for doc in docs)

print(f"\n📊 STATISTICI DOCUMENTE:")
print(f"   Orase: {len(cities)}")
print(f"   Documente RAG: {len(docs)}")
print(f"   Sectiuni detectate: {', '.join(section for section in sections if section)[:220]}...")
print(f"   Total caractere curate: {total_chars:,}")

print("\n📝 EXEMPLU DOCUMENT:")
example = docs[0]
print(f"   Metadata: {example.metadata}")
print(example.page_content[:500])


INCARCARE FISIERE TEXT ALE ORASELOR

📁 Se incarca din: informations\cities_text
🔍 Gasit 315 fisiere cu orase

[1/315] ✓ abrud                     ( 11,156 caractere)
[50/315] ✓ buziaș                    ( 18,726 caractere)
[100/315] ✓ deta                      ( 11,572 caractere)
[150/315] ✓ livada                    (    880 caractere)
[200/315] ✓ petrila                   (  8,856 caractere)
[250/315] ✓ sulina                    (  8,883 caractere)
[300/315] ✓ voluntari                 ( 14,303 caractere)

✓ Incarcat 315 documente cu orase

📊 STATISTICI DOCUMENTE:
   Total caractere: 5,829,530
   Medie caractere/oras: 18,506
   Cel mai mare: 217,873 caractere
   Cel mai mic: 7 caractere


In [5]:
## Pasul 3: Imparte documentele section-aware in bucati

from langchain_text_splitters import RecursiveCharacterTextSplitter

print("\n" + "=" * 100)
print("IMPARTIRE DOCUMENTE IN BUCATI")
print("=" * 100)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1400,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""]
)

print("\n⚙️ Configuratie impartitor:")
print("   Dimensiune bucata: 1400 caractere")
print("   Suprapunere: 150 caractere")
print("   Separatori: paragrafuri, linii noi, propozitii, punct si virgula, virgule, cuvinte, caractere")
print("   Important: chunking-ul se face dupa sectionare, nu pe articolul brut\n")

print("🔪 Se impart documente...")
splits = text_splitter.split_documents(docs)

for chunk_index, chunk in enumerate(splits):
    chunk.metadata['chunk_index'] = chunk_index
    chunk.metadata['chunk_id'] = f"{chunk.metadata.get('city_key', 'unknown')}::{chunk.metadata.get('section', 'section')}::{chunk_index:05d}"

print(f"\n✓ Creat {len(splits)} bucati din {len(docs)} documente curate")

avg_chunk_size = sum(len(chunk.page_content) for chunk in splits) / len(splits) if splits else 0
print(f"\n📊 STATISTICI IMPARTIRE:")
print(f"   Documente originale: {len(docs)}")
print(f"   Total bucati: {len(splits)}")
print(f"   Dimensiune medie bucata: {avg_chunk_size:.0f} caractere")

print("\n📝 EXEMPLE BUCATI:")
for i in range(min(3, len(splits))):
    chunk = splits[i]
    print(f"\n--- Bucata {i+1} ---")
    print(f"Metadata: city_key={chunk.metadata.get('city_key')}, section={chunk.metadata.get('section')}")
    print(chunk.page_content[:350].replace('\n', ' '))



IMPARTIRE DOCUMENTE IN BUCATI

⚙️ Configuratie impartitor:
   Dimensiune bucata: 1000 caractere
   Suprapunere: 100 caractere
   Separatori: paragrafuri, linii noi, propozitii, cuvinte, caractere

🔪 Se impart documente...

✓ Creat 7540 bucati din 315 documente

📊 STATISTICI IMPARTIRE:
   Documente originale: 315
   Total bucati: 7540
   Medie bucati per oras: 23.9
   Dimensiune medie bucata: 795 caractere
   Raport compresie: 23.94x

📝 EXEMPLE BUCATI:
   [1] abrud: AbrudAcest articol se referă la un oraș din județul Alba, România. Pentru o depresiune din Munții Ap...
   [2] abrud: Denumire Încă de la descoperirea Tăblițelor romane cerate de la Roșia Montană în 1786, filologul ger...
   [3] abrud: Geografie Orașul Abrud este situat în depresiunea Abrudului, un spațiu dominat de un relief vălurit,...


In [ ]:
## Pasul 4: Configurare embedding-uri

import requests
from typing import List
from langchain.embeddings.base import Embeddings

print("\n" + "=" * 100)
print("CONFIGURARE EMBEDDING-URI (TRANSFORMARI VECTOR)")
print("=" * 100)

class LocalServerEmbeddings(Embeddings):
    """
    Clasa pentru conectare la serverul local de embedding-uri.
    Se conecteaza la http://localhost:1234/v1/embeddings
    """
    
    def __init__(self, base_url: str = "http://localhost:1234/v1"):
        self.base_url = base_url
        self.model = "text-embedding-nomic-embed-text-v1.5"
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Genereaza embedding-uri pentru o lista de texte (documente)"""
        try:
            response = requests.post(
                f"{self.base_url}/embeddings",
                json={"input": texts}
            )
            response.raise_for_status()
            data = response.json()
            return [item["embedding"] for item in data["data"]]
        except Exception as e:
            print(f"❌ Eroare la embedding: {e}")
            raise
    
    def embed_query(self, text: str) -> List[float]:
        """Genereaza embedding pentru o intrebare (query)"""
        try:
            response = requests.post(
                f"{self.base_url}/embeddings",
                json={"input": [text]}
            )
            response.raise_for_status()
            data = response.json()
            return data["data"][0]["embedding"]
        except Exception as e:
            print(f"❌ Eroare conectare la serverul embedding: {e}")
            print(f"   Verifica: http://localhost:1234/v1/embeddings")
            print(f"   Verifica ca Ollama sau serverul local ruleaza")
            raise

print("\n✓ Definita clasa LocalServerEmbeddings")

print("\n⚙️ Initiare embedding model:")
embeddings = LocalServerEmbeddings(base_url="http://localhost:1234/v1")
print(f"   URL: {embeddings.base_url}/embeddings")
print(f"   Model: {embeddings.model}")

print("\n🧪 Test conexiune - genereaza embedding pentru text de test...")
try:
    test_embedding = embeddings.embed_query("Salut, acesta este un test!")
    print(f"   ✓ Succes! Embedding generat: vector cu {len(test_embedding)} dimensiuni")
    print(f"   ✓ Primii 5 vectori: {[f'{x:.4f}' for x in test_embedding[:5]]}")
except Exception as e:
    print(f"   ❌ Eroare: {e}")


CONFIGURARE EMBEDDING-URI (TRANSFORMARI VECTOR)

✓ Definita clasa LocalServerEmbeddings

⚙️ Initiare embedding model:
   URL: http://localhost:1234/v1/embeddings
   Model: text-embedding-nomic-embed-text-v1.5

🧪 Test conexiune - genereaza embedding pentru text de test...
   ✓ Succes! Embedding generat: vector cu 768 dimensiuni
   ✓ Primii 5 vectori: ['0.0349', '0.0278', '-0.1692', '-0.0153', '0.0481']


In [10]:
## Pasul 5: Creare depozit vector (Vector Store) cu Chroma

import os
import shutil
from langchain_community.vectorstores import Chroma

print("\n" + "=" * 100)
print("CREARE DEPOZIT VECTOR (CHROMA DATABASE)")
print("=" * 100)

# Stergere depozit anterior (daca exista)
chroma_dir = "chroma_cities"
if os.path.exists(chroma_dir):
    print(f"\n🗑️ Sterge depozit anterior: {chroma_dir}")
    shutil.rmtree(chroma_dir)

print(f"\n📦 Configurare Chroma:")
print(f"   Directorul depozitului: {chroma_dir}")
print(f"   Total bucati (chunks) de introdus: {len(splits)}")
print(f"   Model embedding: text-embedding-3-small")

print(f"\n⏳ Se creeaza depozitul vector din {len(splits)} bucati...")
print("   (aceasta operatie poate dura cateva minute...)")

vector_store = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=chroma_dir,
    collection_name="romanian_cities"
)

print(f"\n✓ Depozit vector creat cu succes!")

# Verifica colectia
try:
    collection = vector_store._collection
    count = collection.count()
    print(f"\n📊 STATISTICI DEPOZIT:")
    print(f"   Bucati indexate: {count}")
    print(f"   Directorul depozitului: {chroma_dir}")
    print(f"   Oras de stocare: ~100-200 MB (in functie de modele)")
except Exception as e:
    print(f"   ⚠️ Nu pot verifica numarul exact de bucati: {e}")

print(f"\n✓ Depozitul vector este pregatit pentru cautari semantice!")


CREARE DEPOZIT VECTOR (CHROMA DATABASE)

📦 Configurare Chroma:
   Directorul depozitului: chroma_cities
   Total bucati (chunks) de introdus: 7540
   Model embedding: text-embedding-3-small

⏳ Se creeaza depozitul vector din 7540 bucati...
   (aceasta operatie poate dura cateva minute...)


In [11]:
## Pasul 6: Configurare model Qwen 3.5-2B

from langchain_openai import ChatOpenAI

print("\n" + "=" * 100)
print("CONFIGURARE MODEL QWEN 3.5-2B (LARGE LANGUAGE MODEL)")
print("=" * 100)

print("\n⚙️ Configuratie LLM:")
llm = ChatOpenAI(
    model="qwen",
    temperature=0.7,
    max_tokens=500,
    openai_api_base="http://localhost:1234/v1",
    openai_api_key="not-needed"
)

print(f"   Model: Qwen 3.5-2B")
print(f"   URL Server: http://localhost:1234/v1")
print(f"   Temperatura: 0.7 (creeaza/creativ)")
print(f"   Token-uri maximi: 500")

print(f"\n🧪 Test conexiune la Qwen...")
try:
    test_message = "Salut! Spune-mi intr-o singura linie cine esti."
    test_response = llm.predict(test_message)
    print(f"   ✓ Succes! Qwen raspunde:")
    print(f"   \"{test_response[:100]}...\"")
except Exception as e:
    print(f"   ❌ Eroare conectare la Qwen: {e}")
    print(f"   Verifica: http://localhost:1234/v1/chat/completions")
    print(f"   Verifica ca Ollama / LM Studio ruleaza local")


CONFIGURARE MODEL QWEN 3.5-2B (LARGE LANGUAGE MODEL)

⚙️ Configuratie LLM:
   Model: Qwen 3.5-2B
   URL Server: http://localhost:1234/v1
   Temperatura: 0.7 (creeaza/creativ)
   Token-uri maximi: 500

🧪 Test conexiune la Qwen...
   ❌ Eroare conectare la Qwen: 'ChatOpenAI' object has no attribute 'predict'
   Verifica: http://localhost:1234/v1/chat/completions
   Verifica ca Ollama / LM Studio ruleaza local


In [ ]:
## Pasul 7: Creare lant RAG (Retrieval-Augmented Generation)

from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

print("\n" + "=" * 100)
print("CREARE LANT RAG (RETRIEVAL + GENERATION)")
print("=" * 100)

# Prompt personalizat in romana, adaptat pentru metadata city/section

template_ro = """Esti un asistent factual pentru orase din Romania.
Raspunde in romana folosind exclusiv contextul recuperat.

CONTEXT DIN BAZA DE DATE:
{context}

INTREBAREA UTILIZATORULUI:
{question}

INSTRUCTIUNI:
- Foloseste doar informatiile din context; nu inventa fapte despre calitatea vietii, chirii, salarii, siguranta sau locuri de munca.
- Mentioneaza orasul si sectiunea sursa atunci cand folosesti o informatie, de exemplu [Adjud - Transport].
- Daca o informatie are an, mentioneaza anul.
- Daca datele sunt insuficiente, spune clar ce lipseste.
- Pentru intrebari de relocare, separa faptele prezente in date de informatiile lipsa necesare pentru o recomandare completa.
- Raspunde concis.

RASPUNS:
"""

prompt = PromptTemplate(
    template=template_ro,
    input_variables=["context", "question"]
)

print("\n⚙️ Configuratie lant RAG:")
print("   Retriever: Chroma (cautare semantica metadata-aware)")
print("   LLM: Qwen")
print("   Chain type: RetrievalQA")

retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 6, "fetch_k": 30, "lambda_mult": 0.5})
print("   Cautare: MMR, k=6, fetch_k=30, lambda_mult=0.5")

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

print("\n✓ Lant RAG creat cu succes!")


In [ ]:
## Pasul 8: Testare sistem RAG

print("\n" + "=" * 100)
print("TESTARE SISTEM RAG - INTREBARI ETALON")
print("=" * 100)

def display_rag_result(query: str, result: dict):
    print(f"\n{'='*80}")
    print(f"❓ INTREBARE: {query}")
    print(f"{'='*80}")
    print(f"\n📝 RASPUNS QWEN:")
    print(f"{result['result']}")
    print(f"\n📚 DOCUMENTE SURSA (top relevante):")
    for i, doc in enumerate(result['source_documents'], 1):
        city = doc.metadata.get('city', 'necunoscut')
        section = doc.metadata.get('section', 'necunoscuta')
        preview = doc.page_content[:150].replace('\n', ' ')
        print(f"   [{i}] {city} - {section}: {preview}...")

# Teste etalon
test_queries = [
    "Care este populația din Adjud?",
    "Ce transport are Adjud?",
    "Merită să mă mut în Adjud?"
]

print(f"\n🔄 Ruleaza {len(test_queries)} intrebari de test...\n")

for i, query in enumerate(test_queries, 1):
    try:
        print(f"\n⏳ Test {i}/{len(test_queries)}: Se proceseaza intrebarea...")
        result = qa_chain({"query": query})
        display_rag_result(query, result)
    except Exception as e:
        print(f"❌ Eroare la test {i}: {e}")

print(f"\n{'='*80}")
print("✓ Testare finalizata")


In [ ]:
## Pasul 9: Interfata RAG interactiva

print("\n" + "=" * 100)
print("SISTEM INTERACTIV DE CAUTARE - ORASE ROMANESTI")
print("=" * 100)

def query_cities_rag(question: str, verbose: bool = True):
    """
    Functie interactiva pentru cautare informatii despre orase din Romania.
    
    PARAMETRI:
        question (str): Intrebarea dumneavoastra despre orasele din Romania
        verbose (bool): Afiseaza detalii suplimentare (documente sursa, etc.)
    
    EXEMPLE DE INTREBARI:
        - "Care sunt principalele atractii din Brașov?"
        - "Ce informații are despre Belfast?" (nota: nu e oras roman)
        - "Cine a fondat Sibiul?"
        - "Ce economie are Timisoara?"
        - "Podul Carol din care oras se afla?"
    
    RETUR:
        str: Raspunsul generat de Qwen bazat pe informații din baza de date
    """
    
    if not question or not question.strip():
        return "❌ Eroare: Intrebarea nu poate fi goala!"
    
    try:
        print(f"\n🔍 Cauta informatii despre: '{question}'")
        print("   ⏳ Se interogheaza depozitul vector...")
        
        result = qa_chain({"query": question})
        
        print(f"\n✓ Rezultat gasit!\n")
        print(f"📝 RASPUNS:")
        print(f"{result['result']}\n")
        
        if verbose:
            print(f"📚 DOCUMENTE RELEVANTE (top-3):")
            for i, doc in enumerate(result['source_documents'], 1):
                city = doc.metadata.get('city', 'necunoscut')
                preview = doc.page_content[:100].replace('\n', ' ')
                print(f"   [{i}] {city}: {preview}...")
        
        return result['result']
    
    except Exception as e:
        return f"❌ Eroare la procesare: {str(e)}"

# Mesaj de bun venit
print("\n" + "="*80)
print("✅ SISTEM PREGATIT! Poti pune intrebari despre orasele Romaniei")
print("="*80)

print("\n📌 UTILIZARE:")
print("   query_cities_rag('Intrebarea ta despre orase din Romania')")

print("\n💡 EXEMPLE:")
print("   query_cities_rag('Care sunt atractiile turistice din Sibiu?')")
print("   query_cities_rag('Cum se deplaseaza intre Iasi si Cluj?')")
print("   query_cities_rag('Ce se stie despre istoria Bucurestiului?')")
print("   query_cities_rag('Care este cea mai mare oras din Romania?')")

print("\n🎯 TIP: Pune intrebari specifice pentru rezultate mai bune!")
print("="*80)

## System Architecture & Documentation

### 🏗️ RAG Pipeline Overview

```
City Text Files
    ↓
Wikipedia Cleaner + Section Parser
    ↓
Structured City JSON + Section Documents
    ↓
RecursiveCharacterTextSplitter (1400 chars, 150 overlap)
    ↓
Embeddings
    ↓
Chroma Vector Database
    ↓
Metadata-aware retrieval (city_key, county, section)
    ↓
Qwen answer generation
```

### ⚙️ Key Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| **chunk_size** | 1400 | Characters per chunk |
| **chunk_overlap** | 150 | Overlap between chunks |
| **top_k** | 5-6 | Documents to retrieve |
| **temperature** | 0.0-0.2 | Low temperature for factual RAG |
| **metadata filters** | city_key, county, section | Used for city and topic specific queries |

### 🔍 Retrieval Process

- Use MMR for broad semantic search to reduce near-duplicate chunks.
- Use `city_key` filters when the query clearly names a city.
- Use section filters for obvious topics: Demografie, Transport, Educație, Sănătate, Istorie, Geografie.
- Keep source attribution visible as `[Oraș - Secțiune]`.

### 📋 Supported Queries

✓ Geographic information  
✓ Historical facts  
✓ Demographics and population by year  
✓ Transport roads and railway facts  
✓ Education and healthcare facts when present  
✓ Relocation questions with explicit missing-data caveats
